# Loading openff-pablo's prepared PDB corpus

Every file in Pablo's `prepared_pdbs` test set, loaded with `Protein`.
Templates outside the 34 shipped components download from the RCSB.

In [1]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [2]:
# Pablo's own test corpus of prepared PDB files, fetched into a git-ignored cache.
import urllib.request
from pathlib import Path

CORPUS_URL = "https://raw.githubusercontent.com/openforcefield/openff-pablo/main/openff/pablo/_tests/data/prepared_pdbs/"
CORPUS_FILES = [
    "193l_prepared.pdb",
    "1FLR_prepared.pdb",
    "1a4t_samechain.pdb",
    "1csa_maestro.pdb",
    "1csa_maestro_waterfirst.pdb",
    "1hje_diffchain.pdb",
    "1hje_samechain.pdb",
    "1p3q_noter.pdb",
    "2MUM_blowup.pdb",
    "2MUM_composed_function.pdb",
    "2MUM_discontiguous_resseq.pdb",
    "2MUM_discontiguous_serial.pdb",
    "2MUM_dryrun.pdb",
    "2MUM_icode.pdb",
    "2MUM_letters_in_resseq.pdb",
    "2MUM_letters_in_serial.pdb",
    "2MUM_neutralized.pdb",
    "2MUM_reuse_resseq.pdb",
    "2MUM_reuse_serial.pdb",
    "2hi7_prepared.pdb",
    "2zuq_prepared.pdb",
    "3h34_prepared.pdb",
    "3ip9_dye_solvated.pdb",
    "5eil_fixed.pdb",
    "ions.pdb",
    "polyglycines.pdb"
]
cache = Path("../assets_cache/pablo_prepared_pdbs")
cache.mkdir(parents=True, exist_ok=True)
for name in CORPUS_FILES:
    if not (cache / name).exists():
        urllib.request.urlretrieve(CORPUS_URL + name, cache / name)
len(list(cache.glob("*.pdb"))), "files"

(26, 'files')

In [3]:
from mbuild.biopolymers import Protein

loaded, failed = [], []
for path in sorted(cache.glob("*.pdb")):
    try:
        protein = Protein(path, download=True)
    except Exception as error:
        failed.append((path.name, str(error).splitlines()[0]))
        continue
    loaded.append(path.name)
    print(
        f"{path.name:32s} {len(protein.chains):3d} chains"
        f" {len(list(protein.residues())):5d} residues"
        f" {protein.n_particles:6d} atoms"
        f"  net charge {protein.net_formal_charge:+d}"
        f"  crosslinks {len(protein.bond_records())}"
    )
print()
print(len(loaded), "loaded,", len(failed), "refused")

193l_prepared.pdb                  1 chains   129 residues   1960 atoms  net charge +8  crosslinks 4


1p3q_noter.pdb                     4 chains   228 residues   3625 atoms  net charge -11  crosslinks 0
2MUM_blowup.pdb                    1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_composed_function.pdb         1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_discontiguous_resseq.pdb      1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_discontiguous_serial.pdb      1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_dryrun.pdb                    1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_icode.pdb                     1 chains    50 residues    795 atoms  net charge +0  crosslinks 0


2MUM_letters_in_resseq.pdb         1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_letters_in_serial.pdb         1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_neutralized.pdb               1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_reuse_resseq.pdb              1 chains    50 residues    795 atoms  net charge +0  crosslinks 0
2MUM_reuse_serial.pdb              1 chains    50 residues    795 atoms  net charge +0  crosslinks 0


2hi7_prepared.pdb                  2 chains   322 residues   5134 atoms  net charge -4  crosslinks 2


2zuq_prepared.pdb                  4 chains   730 residues  11283 atoms  net charge +0  crosslinks 7
3h34_prepared.pdb                  2 chains   176 residues   1624 atoms  net charge -4  crosslinks 0


5eil_fixed.pdb                     6 chains   582 residues   7491 atoms  net charge -15  crosslinks 0
ions.pdb                          26 chains   565 residues   1675 atoms  net charge +0  crosslinks 0

18 loaded, 8 refused


Each refusal names the residue and says why.

In [4]:
for name, reason in failed:
    print(f"{name:32s} {reason}")

1FLR_prepared.pdb                Could not match residue ASP H:76 against any template variant:
1a4t_samechain.pdb               Could not match residue G A:4 against any template variant:
1csa_maestro.pdb                 Could not match residue DAL A:0 against any template variant:
1csa_maestro_waterfirst.pdb      Could not match residue DAL A:0 against any template variant:
1hje_diffchain.pdb               Could not match residue CYS A:13 against any template variant:
1hje_samechain.pdb               Could not match residue CYS A:13 against any template variant:
3ip9_dye_solvated.pdb            Residue DYE A:178: Could not download CCD definition for 'DYE': HTTP Error 404: Not Found Pass download=True to fetch unknown residues from RCSB.
polyglycines.pdb                 Could not match residue GLY A:1 against any template variant:


Pablo's verdict on the same eight files, for comparison. Three of them Pablo
refuses too without extra input. One is DNA, which `Protein` does not claim.
The other four are two copies each of a cyclic peptide (`1csa`) and of a
peptide with a C-terminal `NH2` cap (`1hje`). Pablo handles both and
`Protein` does not yet.

In [5]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

STD_CCD_CACHE.auto_download = True
for name, _ in failed:
    try:
        topology = topology_from_pdb(cache / name)
        print(f"{name:32s} pablo loads it, {topology.n_atoms} atoms")
    except Exception as error:
        print(f"{name:32s} pablo refuses it too: {type(error).__name__}")

1FLR_prepared.pdb                pablo refuses it too: PdbResidueMatchError
1a4t_samechain.pdb               pablo loads it, 836 atoms


1csa_maestro.pdb                 pablo loads it, 6225 atoms


1csa_maestro_waterfirst.pdb      pablo loads it, 6225 atoms
1hje_diffchain.pdb               pablo loads it, 256 atoms
1hje_samechain.pdb               pablo loads it, 256 atoms


3ip9_dye_solvated.pdb            pablo refuses it too: PdbResidueMatchError
polyglycines.pdb                 pablo refuses it too: PdbResidueMatchError
